# Machine Translation Exercise
---

In [117]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import datasets
from tqdm import tqdm
import spacy
from pprint import pprint
from collections import Counter
import json
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

## Prepare Data and Preprocessing

In [2]:
# Read sentences from text files and put them to a dataframe
swedish_sentences = []
english_sentences = []
with open('./translation_data/microlang_20000_swe.txt') as file:
    swedish_sentences = file.readlines()
with open('./translation_data/microlang_20000_eng.txt') as file:
    english_sentences = file.readlines()
for i, s in enumerate(swedish_sentences):
    swedish_sentences[i] = s.strip()
for i, s in enumerate(english_sentences):
    english_sentences[i] = s.strip()
sentence_df = pd.DataFrame({
    'swe': swedish_sentences,
    'eng': english_sentences
})
sentence_df

,swe,eng
0,stora hav är dåliga,big seas are bad
1,de går,they walk
2,ån går,the river walks
3,de heta kvinnorna dödar dem nu,the hot women kill them now
4,hästar går,horses go
...,...,...
19995,han är gul,he is yellow
19996,hon slår dem sönder,she breaks them
19997,hon äter henne redan,she eats her already
19998,flickorna slår henne sönder med henne,the girls break her with her


In [3]:
# Create and split the dataset
# Split the whole dataset to get train data first, then split the big test data into small validation and test data
unsplitted_dataset = datasets.Dataset.from_pandas(sentence_df, split='train')
train_test_dataset = unsplitted_dataset.train_test_split(test_size=0.15, seed=42)
val_test_dataset = train_test_dataset['test'].train_test_split(test_size=0.5, seed=42)
dataset = datasets.DatasetDict({
    'train': train_test_dataset['train'],
    'validation': val_test_dataset['train'],
    'test': val_test_dataset['test']
})
dataset

DatasetDict({
    train: Dataset({
        features: ['swe', 'eng'],
        num_rows: 17000
    })
    validation: Dataset({
        features: ['swe', 'eng'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['swe', 'eng'],
        num_rows: 1500
    })
})

In [4]:
dataset['train'][:5]

{'swe': ['han simmar',
  'den färdiga hästen kommer',
  'han hoppar',
  'han sover',
  'de kalla kvinnorna är blåa'],
 'eng': ['he swims',
  'the ready horse comes',
  'he jumps',
  'he sleeps',
  'the cold women are blue']}

In [5]:
# Define special tokens and parameters
max_length = 100
lower = True
sos_token = "<sos>"
eos_token = "<eos>"
unk_token = "<unk>"
pad_token = "<pad>"
min_freq = 2
special_tokens = [unk_token, pad_token, sos_token, eos_token]

To install tokenizer for English and Swedish:

`python -m spacy download en_core_web_sm`

`python -m spacy download sv_core_news_sm`

In [6]:
# Define tokenizer
en_nlp = spacy.load('en_core_web_sm')
sv_nlp = spacy.load('sv_core_news_sm')

In [7]:
def process_sentence(sentence, en_vocab, sv_vocab):
    # Tokenize English and Swedish
    en_tokens = [token.text for token in en_nlp.tokenizer(sentence["eng"])][:max_length]
    sv_tokens = [token.text for token in sv_nlp.tokenizer(sentence["swe"])][:max_length]  

    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        sv_tokens = [token.lower() for token in sv_tokens]

    # Add special tokens
    en_tokens = [sos_token] + en_tokens + [eos_token]
    sv_tokens = [sos_token] + sv_tokens + [eos_token]

    # Numericalize tokens
    en_ids = [en_vocab.get(token, en_vocab[unk_token]) for token in en_tokens]
    sv_ids = [sv_vocab.get(token, sv_vocab[unk_token]) for token in sv_tokens]

    return {
        "en": sentence["eng"],
        "sv": sentence["swe"],
        "en_tokens": en_tokens,
        "sv_tokens": sv_tokens,
        "en_ids": en_ids,
        "sv_ids": sv_ids,
    }

In [8]:
# Step 2: Build Vocabulary
def build_vocab(data, min_freq, specials):
    counter = Counter()
    for tokens in data:
        counter.update(tokens)
    vocab = {token: idx for idx, token in enumerate(specials)}
    idx = len(vocab)
    for token, freq in counter.items():
        if freq >= min_freq and token not in vocab:
            vocab[token] = idx
            idx += 1
    return vocab

In [9]:
# Step 3: Generate Tokenized Data for Vocabulary Building
tokenized_train_data = dataset['train'].map(
    lambda example: {"en_tokens": [token.text for token in en_nlp.tokenizer(example["eng"])][:max_length],
                     "sv_tokens": [token.text for token in sv_nlp.tokenizer(example["swe"])][:max_length]}
)

Map: 100%|██████████| 17000/17000 [00:01<00:00, 15392.07 examples/s]


In [10]:
# Build vocabularies
en_vocab = build_vocab(tokenized_train_data["en_tokens"], min_freq, special_tokens)
sv_vocab = build_vocab(tokenized_train_data["sv_tokens"], min_freq, special_tokens)

In [11]:
# Step 4: Process Full Dataset
train_data = dataset['train'].map(lambda x: process_sentence(x, en_vocab, sv_vocab))
valid_data = dataset['validation'].map(lambda x: process_sentence(x, en_vocab, sv_vocab))
test_data = dataset['test'].map(lambda x: process_sentence(x, en_vocab, sv_vocab))

Map: 100%|██████████| 1500/1500 [00:00<00:00, 14371.04 examples/s]


In [12]:
# Save vocabularies to JSON
with open("en_vocab.json", "w") as f:
    json.dump(en_vocab, f)
with open("sv_vocab.json", "w") as f:
    json.dump(sv_vocab, f)

In [13]:
# Check for special tokens in both vocabularies - this is debugging
assert en_vocab[unk_token] == sv_vocab[unk_token]
assert en_vocab[pad_token] == sv_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [14]:
data_type = "torch"
format_columns = ["en_ids", "sv_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

In [15]:
print(train_data)
pprint(train_data[1])

Dataset({
    features: ['swe', 'eng', 'en', 'sv', 'en_tokens', 'sv_tokens', 'en_ids', 'sv_ids'],
    num_rows: 17000
})
{'en': 'the ready horse comes',
 'en_ids': tensor([2, 6, 7, 8, 9, 3]),
 'en_tokens': ['<sos>', 'the', 'ready', 'horse', 'comes', '<eos>'],
 'eng': 'the ready horse comes',
 'sv': 'den färdiga hästen kommer',
 'sv_ids': tensor([2, 6, 7, 8, 9, 3]),
 'sv_tokens': ['<sos>', 'den', 'färdiga', 'hästen', 'kommer', '<eos>'],
 'swe': 'den färdiga hästen kommer'}


## Data Loaders

In [16]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_sv_ids = [example["sv_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index, batch_first=True)
        batch_sv_ids = nn.utils.rnn.pad_sequence(batch_sv_ids, padding_value=pad_index, batch_first=True)
        batch = {
            "en_ids": batch_en_ids,
            "sv_ids": batch_sv_ids,
        }
        return batch

    return collate_fn

In [17]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [18]:
batch_size = 64

train_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_loader = get_data_loader(valid_data, batch_size, pad_index)
test_loader = get_data_loader(test_data, batch_size, pad_index)

## Define the Encoder and Decoder for Seq2Seq 

### Encoder

- Encoder reads the input sequence and summerizes the information in something called internal state vectors or context vectors. This context vector aims to encapsulate the information for all input elements to help the decoder make accurate predictions.
- This implementation involves creating an RNN-based encoder.

In [340]:
# Here, define class Encoder with __init__ and forward functions
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super(Encoder, self).__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
    
    def forward(self, x):
        x = x.transpose(0, 1)
        x = self.embedding(x)
        _, (hidden, cell) = self.lstm(x)
        
        return hidden, cell

### Decoder

In [341]:
# Similarly, introduce class Decoder here
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super(Decoder, self).__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.output_dim = output_dim

        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.linear = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x, hidden, cell):
        x = self.embedding(x.unsqueeze(0))
        output, (hidden, cell) = self.lstm(x, (hidden, cell))
        output = self.linear(output.squeeze(0))

        return output, (hidden, cell)

### Seq2Seq Model 

For the final part of the implemenetation, we'll implement the seq2seq model. This will handle:

- receiving the input/source sentence
- using the encoder to produce the context vectors
- using the decoder to produce the predicted output/target sentence

In [342]:
#Here, introduce a Seq2Seq class using the encoder and the decoder
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[:, 0] # Passing the starting token (SOS)
        for t in range(1, trg_len):
            output, (hidden, cell) = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output # Predictions upto target length
            teacher_force = random.random() < teacher_forcing_ratio # 50% of the time we use reference tokens instead of predicted tokens
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
        return outputs

In [343]:
input_dim = len(sv_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5

if torch.mps.is_available():
    device = torch.device('mps')
elif torch.xpu.is_available():
    device = torch.device('xpu')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device).to(device)

In [344]:
print("Input Dim (Swedish Vocab):", input_dim)
print("Output Dim (English Vocab):", output_dim)

Input Dim (Swedish Vocab): 252
Output Dim (English Vocab): 165


In [345]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.01, 0.01)

model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(252, 256)
    (lstm): LSTM(256, 512, num_layers=2, dropout=0.5)
  )
  (decoder): Decoder(
    (embedding): Embedding(165, 256)
    (lstm): LSTM(256, 512, num_layers=2, dropout=0.5)
    (linear): Linear(in_features=512, out_features=165, bias=True)
  )
)

In [346]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 7,547,813 trainable parameters


In [347]:
assert encoder.hidden_dim == decoder.hidden_dim
assert encoder.n_layers == decoder.n_layers

In [ ]:
# This part is only for computing BLEU score
bleu_weights = (1/3, 1/3, 1/3)
special_token_ids = {0, 1, 2, 3}
def remove_special_tokens(tensor):
    tensor_list = [[v for v in row.tolist() if v not in special_token_ids] for row in tensor]
    for row in tensor_list:
        # If sentence length is smaller than number n in n-grams, add a fixed word until the length = n
        # It makes, for example, 2-word sentence with correct translation get the score 1 instead of very low score
        if len(row) < len(bleu_weights):
            for _ in range(len(row), len(bleu_weights)):
                row.append(1000)
    return tensor_list

In [349]:
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio
):
    model.train()
    epoch_loss = 0
    bleu_scores = []
    for i, batch in enumerate(data_loader):
        src = batch["sv_ids"].to(device)
        trg = batch["en_ids"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        # Compute BLEU score for this batch
        output_ids = torch.argmax(output, dim=2)
        hyps = remove_special_tokens(output_ids)
        refs = remove_special_tokens(trg)
        list_of_references = [[r] for r in refs]
        bleu_scores.append(corpus_bleu(list_of_references, hyps, weights=bleu_weights))
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader), np.mean(bleu_scores)

In [350]:
def evaluate_fn(model, data_loader, criterion):
    model.eval()
    epoch_loss = 0
    bleu_scores = []
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["sv_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # Compute BLEU score for this batch
            output_ids = torch.argmax(output, dim=2)
            hyps = remove_special_tokens(output_ids)
            refs = remove_special_tokens(trg)
            list_of_references = [[r] for r in refs]
            bleu_scores.append(corpus_bleu(list_of_references, hyps, weights=bleu_weights))
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader), np.mean(bleu_scores)

In [351]:
n_epochs = 30
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm(range(n_epochs)):
    train_loss, train_mean_bleu = train_fn(
        model,
        train_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,

    )
    valid_loss, valid_mean_bleu = evaluate_fn(
        model,
        valid_loader,
        criterion,

    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "swe-eng-translation-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f} | Train mean BLEU score: {train_mean_bleu:7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f} | Valid mean BLEU score: {valid_mean_bleu:7.3f}")

  3%|▎         | 1/30 [00:07<03:41,  7.63s/it]

	Train Loss:   3.447 | Train PPL:  31.420 | Train mean BLEU score:   0.002
	Valid Loss:   3.168 | Valid PPL:  23.752 | Valid mean BLEU score:   0.004


  7%|▋         | 2/30 [00:14<03:28,  7.46s/it]

	Train Loss:   2.841 | Train PPL:  17.135 | Train mean BLEU score:   0.009
	Valid Loss:   2.768 | Valid PPL:  15.931 | Valid mean BLEU score:   0.015


 10%|█         | 3/30 [00:22<03:21,  7.47s/it]

...

	Train Loss:   0.931 | Train PPL:   2.538 | Train mean BLEU score:   0.942
	Valid Loss:   0.955 | Valid PPL:   2.597 | Valid mean BLEU score:   0.912


 97%|█████████▋| 29/30 [03:35<00:07,  7.43s/it]

	Train Loss:   0.925 | Train PPL:   2.522 | Train mean BLEU score:   0.959
	Valid Loss:   0.947 | Valid PPL:   2.579 | Valid mean BLEU score:   0.926


100%|██████████| 30/30 [03:43<00:00,  7.44s/it]

	Train Loss:   0.917 | Train PPL:   2.501 | Train mean BLEU score:   0.962
	Valid Loss:   0.940 | Valid PPL:   2.560 | Valid mean BLEU score:   0.927


## Save and test model

In [352]:
model.load_state_dict(torch.load("swe-eng-translation-model.pt"))

test_loss, test_mean_bleu = evaluate_fn(model, test_loader, criterion)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} | Test mean BLEU score: {test_mean_bleu:7.3f}")

| Test Loss: 0.943 | Test PPL:   2.567 | Test mean BLEU score:   0.922


In [353]:
def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_length=50):
    model.eval()
    
    # Tokenize the sentence
    tokens = [token.text.lower() for token in sv_nlp(sentence)]
    # Add <sos> and <eos> tokens
    tokens = [sos_token] + tokens + [eos_token]
    # Convert tokens to indices
    src_indexes = [src_vocab.get(token, src_vocab[unk_token]) for token in tokens]
    # Convert to tensor and add batch dimension
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(device)
    
    # Encode the source sentence
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
    
    # Initialize the target sentence with <sos> token
    trg_indexes = [trg_vocab[sos_token]]
    
    for _ in range(max_length):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, (hidden, cell) = model.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        
        if pred_token == trg_vocab[eos_token]:
            break
    
    trg_tokens = [list(trg_vocab.keys())[list(trg_vocab.values()).index(i)] for i in trg_indexes]
    
    return trg_tokens[1:-1]

In [ ]:
# Test the model on test data
test_idx = 21
sentence = test_data[test_idx]['sv']
expected_result = test_data[test_idx]['en']
translation = translate_sentence(sentence, sv_vocab, en_vocab, model, device)

print("Original sentence:", sentence)
print("Translated sentence:", " ".join(translation))
print("Expected result:", expected_result)

Original sentence: en god fågel slår henne sönder redan
Translated sentence: a good bird breaks her already
Expected result: a good bird breaks her already


In [ ]:
def calculate_bleu_score(reference, candidate):
    """
    Calculate BLEU score for a single reference and candidate sentence pair.
    
        :param reference: List of words in the target sentence (ground truth).
        :param candidate: List of words in the predicted sentence.

    Return: BLEU score (float)
    """
    # If sentence length is smaller than number n in n-grams, add a fixed word until the length = n
    # It makes, for example, 2-word sentence with correct translation get the score 1 instead of very low score
    if len(reference) < len(bleu_weights):
        for _ in range(len(reference), len(bleu_weights)):
            reference.append('you')
    if len(candidate) < len(bleu_weights):
        for _ in range(len(candidate), len(bleu_weights)):
            candidate.append('you')
    return sentence_bleu([reference], candidate, weights=bleu_weights)

In [444]:
x = calculate_bleu_score(expected_result.split(), translation)
print('BLEU score:', x)

BLEU score: 1.0


In [ ]:
# Demonstrate the calculation when a 2-word sentence is translated correctly
x2 = calculate_bleu_score(['i', 'can'], ['i', 'can']) # or: x2 = calculate_bleu_score([1, 2], [1, 2])
print('BLEU score:', x2)

BLEU score: 1.0
